In [38]:
import importlib
import auxiliar
importlib.reload(auxiliar)

<module 'auxiliar' from 'c:\\Users\\adria\\Desktop\\Master IA\\AA TFM\\FineTunning\\model\\auxiliar.py'>

In [39]:
API_URL = "https://undauntedly-unchristened-connie.ngrok-free.dev/"
datos_usuario = {}
historial = {}

DATOS BASICOS

In [40]:
print("👩‍⚕️ Hola, soy Sana, tu asistente médico. Vamos a empezar con unas preguntas básicas.")
datos_usuario["nombre"] = input("¿Cuál es tu nombre?")
datos_usuario["edad"] = (input("¿Qué edad tienes?"))
datos_usuario["genero"] = input("¿Cuál es tu sexo biológico?")
datos_usuario["poblacion"] = input("¿Dónde vives?")

print(f"\nGracias, {datos_usuario['nombre']}. Ahora cuéntame qué síntomas presentas:")
sintomas = input("Tú: ")

👩‍⚕️ Hola, soy Sana, tu asistente médico. Vamos a empezar con unas preguntas básicas.

Gracias, adrian ramos olive. Ahora cuéntame qué síntomas presentas:


DIAGNOSTICO PRELIMINAR

In [41]:
data = {
    "question": sintomas,
    "age": datos_usuario["edad"],
    "gender": datos_usuario["genero"]
}
print("\nAnalizando tus síntomas...")

diagnostico_prelim = auxiliar.obtener_respuesta_api(data, API_URL + "diagnostico-preliminar")
print(f"\n{diagnostico_prelim}")


Analizando tus síntomas...
URL: https://undauntedly-unchristened-connie.ngrok-free.dev/diagnostico-preliminar
Data: {'question': 'me duele la garganta, me cuesta tragar', 'age': '23', 'gender': 'masculino'}

{'answer': {'query': 'Paciente masculino, 23 años. Síntomas: me duele la garganta, me cuesta tragar', 'result': '{\n  "Nivel de urgencia": "Bajo",\n  "Posibles diagnósticos": [\n    "Faringitis viral (cuadro gripal/IVRS)",\n    "Faringitis bacteriana (estreptocócica)",\n    "Reflujo gastroesofágico con irritación faríngea",\n    "Absceso periamigdalino (complicación de amigdalitis; a descartar si aparecen signos de alarma)"\n  ],\n  "Justificación": "En un varón joven con odinofagia (dolor al tragar) y dolor de garganta, la causa más frecuente es una infección viral de vías respiratorias superiores, a menudo acompañada de síntomas de “sintomatología gripal” y “congestión nasal” (ver citas). La faringitis estreptocócica es otra posibilidad cuando hay fiebre alta, exudados amigdalar

PREGUNTAS ACLARATORIAS

In [42]:
faltas = auxiliar.imprimir_campo(diagnostico_prelim, "Preguntas aclaratorias para el paciente")
print(faltas)
print("Voy a hacerle unas preguntas acerca de tus síntomas para afinar el diagnóstico.")
# Preguntas aclaratorias
for pregunta in faltas:
    print(pregunta)
    respuesta = input("Tú: ")
    historial[pregunta] = respuesta

# Función para convertir historial a texto médico
def historial_a_texto(historial):
    partes = []
    for pregunta, respuesta in historial.items():
        partes.append(f"PREGUNTA: {pregunta}\nRESPUESTA: {respuesta}")
    return "\n\n".join(partes)

historial_texto = historial_a_texto(historial)



['¿Desde cuándo comenzaron los síntomas y cómo han evolucionado?', '¿Ha tenido fiebre? ¿De cuánto (número en °C)?', '¿Tiene tos, congestión nasal o rinorrea, dolores musculares o cefalea?', '¿El dolor es unilateral? ¿Nota voz apagada/gangosa, dificultad para abrir la boca (trismus) o babeo/dificultad para tragar saliva?', '¿Dificultad para respirar, erupción cutánea o rigidez de nuca?', '¿Contacto reciente con personas enfermas o brotes en su entorno?', '¿Pirosis/ardor, regurgitación ácida, síntomas que empeoran al acostarse o tras comidas (sugestivos de reflujo)?', '¿Náuseas o vómitos, o sialorrea notable?', '¿Ha tomado analgésicos/antiinflamatorios y ha mejorado? ¿Alergias conocidas?', '¿Tabaquismo, alcohol, sexo oral reciente u otras exposiciones relevantes?']
Voy a hacerle unas preguntas acerca de tus síntomas para afinar el diagnóstico.
¿Desde cuándo comenzaron los síntomas y cómo han evolucionado?
¿Ha tenido fiebre? ¿De cuánto (número en °C)?
¿Tiene tos, congestión nasal o rinorr

DIAGNOSTICO FINAL

In [43]:
import json

historial_texto = historial_a_texto(historial)
# Reevaluación (en base a nuevas respuestas)
diagnostico_prelim["answer"]["clarifications"] = historial_texto
diagnostico_final_texto = json.dumps(diagnostico_prelim, ensure_ascii=False, indent=2)

data = {
    "question": diagnostico_final_texto,
    "age": datos_usuario["edad"],
    "gender": datos_usuario["genero"]
}

print("\n🔍 Analizando tus respuestas...")
nuevo_diag = auxiliar.obtener_respuesta_api(data, API_URL + "diagnostico-final")
print(f"{nuevo_diag}")


🔍 Analizando tus respuestas...
URL: https://undauntedly-unchristened-connie.ngrok-free.dev/diagnostico-final
Data: {'question': '{\n  "answer": {\n    "query": "Paciente masculino, 23 años. Síntomas: me duele la garganta, me cuesta tragar",\n    "result": "{\\n  \\"Nivel de urgencia\\": \\"Bajo\\",\\n  \\"Posibles diagnósticos\\": [\\n    \\"Faringitis viral (cuadro gripal/IVRS)\\",\\n    \\"Faringitis bacteriana (estreptocócica)\\",\\n    \\"Reflujo gastroesofágico con irritación faríngea\\",\\n    \\"Absceso periamigdalino (complicación de amigdalitis; a descartar si aparecen signos de alarma)\\"\\n  ],\\n  \\"Justificación\\": \\"En un varón joven con odinofagia (dolor al tragar) y dolor de garganta, la causa más frecuente es una infección viral de vías respiratorias superiores, a menudo acompañada de síntomas de “sintomatología gripal” y “congestión nasal” (ver citas). La faringitis estreptocócica es otra posibilidad cuando hay fiebre alta, exudados amigdalares y ausencia de tos. 

EVALUACION DEL TRIAJE

In [44]:
!pip install fpdf2 -q

In [47]:
importlib.reload(auxiliar)
print(nuevo_diag)
nivel_texto = auxiliar.imprimir_campo(nuevo_diag, "Nivel de urgencia actual (Bajo, Medio, Alto)")
texto = nivel_texto.lower()
print(f"Nivel de urgencia obtenido: {nivel_texto}")

# Clasificación simple basada en palabras clave
if any(palabra in texto for palabra in ["bajo"]):
    # Obtener solo la parte textual del diagnóstico definitivo
    parsed_diag = json.loads(nuevo_diag["answer"]["result"])
    diagnostico_raw = parsed_diag["Diagnóstico definitivo"]
    # 🧹 Normalizar a string
    if isinstance(diagnostico_raw, list):
        diagnostico_str = " ".join(diagnostico_raw)
    else:
        diagnostico_str = str(diagnostico_raw)
    medicamento = auxiliar.medicamentos(datos_usuario, diagnostico_str, API_URL + "medicamentos")
    print(f"Medicamento recomendado: {medicamento}")
    importlib.reload(auxiliar)
    auxiliar.guardar_pdf(datos_usuario, historial, nuevo_diag, medicamento)
elif any(palabra in texto for palabra in ["medio"]):
    importlib.reload(auxiliar)
    auxiliar.guardar_pdf(datos_usuario, historial, nuevo_diag, "Agendar cita con su médico de confianza para mas pruebas.")
    print("medio")
elif any(palabra in texto for palabra in ["alto"]):
    importlib.reload(auxiliar)
    auxiliar.guardar_pdf(datos_usuario, historial, nuevo_diag, "Acudir a urgencias inmediatamente. Debería llamar a una ambulancia urgnentemente.")
    print("alto")
else:
    auxiliar.guardar_pdf(datos_usuario, historial, nuevo_diag, None)
    print("indeterminado")

{'answer': {'query': 'Paciente masculino, 23 años. Información clínica: {\n  "answer": {\n    "query": "Paciente masculino, 23 años. Síntomas: me duele la garganta, me cuesta tragar",\n    "result": "{\\n  \\"Nivel de urgencia\\": \\"Bajo\\",\\n  \\"Posibles diagnósticos\\": [\\n    \\"Faringitis viral (cuadro gripal/IVRS)\\",\\n    \\"Faringitis bacteriana (estreptocócica)\\",\\n    \\"Reflujo gastroesofágico con irritación faríngea\\",\\n    \\"Absceso periamigdalino (complicación de amigdalitis; a descartar si aparecen signos de alarma)\\"\\n  ],\\n  \\"Justificación\\": \\"En un varón joven con odinofagia (dolor al tragar) y dolor de garganta, la causa más frecuente es una infección viral de vías respiratorias superiores, a menudo acompañada de síntomas de “sintomatología gripal” y “congestión nasal” (ver citas). La faringitis estreptocócica es otra posibilidad cuando hay fiebre alta, exudados amigdalares y ausencia de tos. El reflujo gastroesofágico puede irritar la faringe y prod